# Генератор версии 1.0 

Синтез данных для НИРа магистра Ефимовой Анасатсии Сергеевны группы 6133

Основная суть синтеза, на основе численного моделирования из вкр бакалавка синтезировать набор входных параметров (входного сигнала, фильтров), на их основе промоделировать выходной сигнал и выделить признаки выходного сигнала. А далее многометочный классификатор будут по признакам выходного сигнала ~~гадать~~ востанавливать набор используемых типов фильтров. А последующие регрессоры будут востанавливать параметры входного сигнала и фильтров

### Генерируемые данные:

1. Случайно выбираемые параметры

- `alpha_rad_per_mm` - параметр синусоидальной модуляции входного сигнала.
- `sigma_mm` - ширина гауссовой огибающей входного сигнала.
- `slit_target_transmission` - заданная доля мощности спектра, которую должна пропускать щель.
- `period_binary_mm` - период бинарного амплитудного фильтра.
- `duty_cycle` - доля периода бинарного фильтра, в течение которой амплитуда принимает высокое значение.
- `binary_high` - высокое значение амплитуды бинарного фильтра.
- `binary_low` - низкое значение амплитуды бинарного фильтра.
- `phase_slope_rad_per_mm` - наклон линейной фазы.
- `period_phase_sinusoidal_mm` - период синусоидальной фазовой маски.
- `phase_depth_rad` - глубина фазовой модуляции синусоидальной маски.

Параметры фильтров выбираются только тогда, когда соответствующий фильтр включён в комбинацию. Для невключёных фильтров их параметры не используются и могут быть записаны как NaN.

### 2. Вычисляемые данные

- `slit_width_mm` - ширина щели, выбранная на основе заданной целевой пропускаемой мощности спектра.
- `slit_actual_transmission` - фактическая доля мощности спектра, прошедшая через щель.
- `spectral_power_before_slit` - мощность спектра до применения щели.
- `spectral_power_after_slit` - мощность спектра после применения щели.

Для каждого примера также рассчитываются комплексное выходное поле и спектр после фильтрации. Спектр сначала вычисляется с помощью прямого преобразования Фурье, затем к нему применяются выбранные фильтры. После этого обратное преобразование Фурье используется для получения выходного поля. Выходное поле рассчитывается на расширенной пространственной сетке, после чего из него вырезается центральная область для сохранения. Это позволяет сохранить выбранный диапазон выходных координат с более подробным спектральным расчётом при обратном преобразовании.

В датасете сохраняются действительная и мнимая части выходного поля, а также амплитуда и фаза отфильтрованного спектра.

### 3. Признаки выходного сигнала

Для каждого выходного поля рассчитываются следующие признаки:

- `output_energy` - энергия выходного поля на сохранённом пространственном интервале.
- `output_centroid_mm` - координата центра распределения по интенсивности.
- `output_rms_width_mm` - среднеквадратичная ширина распределения относительно его центра.
- `output_peak_intensity` - максимальная интенсивность выходного распределения.
- `output_intensity_symmetry_error` - показатель отличия распределения интенсивности от симметричного относительно центра координат.

Эти признаки позволяют описывать выходное распределение несколькими числами, не анализируя каждый отсчёт поля отдельно. Но эти признаки общие для всех типов фильтрации. Уникальные параметры для фильтров будут расчитываться или вычисляться в следующий версиях генераторов. Это будет необходимо для регрессоров.

### 4. Файлы датасета

- `dataset_params.csv` - таблица параметров и признаков. Каждая строка соответствует одному примеру. В ней хранятся идентификатор примера, комбинация фильтров, метки фильтров, параметры, вычисляемые значения и признаки выходного распределения.
- `dataset_vectors.npz` - массивы с численными данными. В файле хранятся метки фильтров, действительная и мнимая части выходного поля, амплитуда и фаза отфильтрованного спектра, координатные сетки, целевая и фактическая пропускаемая мощность, а также мощности спектра до и после щели.
- `dataset_config.json` - настройки генерации: случайное начальное значение, количество примеров, размеры сеток, физические параметры модели, диапазоны случайных параметров и порядок меток фильтров.

Таким образом, CSV-файл удобно использовать для анализа параметров и признаков, NPZ-файл - для обучения нейронной сети (регрессоров) на полных численных распределениях, а JSON-файл - для сохранения настроек генерации датасета.


## Библиотеки, важные функции и подготовка к генератору

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

In [2]:
# Параметры сиситемы

Wavelength = 0.000633
Focal_length = 100.0
Wave_number = 2 * np.pi / Wavelength

# Сетки

N_input = 2049
N_spectral = 16385
N_output = 9217

X_input_half_width_mm = 1.0
U_half_width_mm = 5.5
X_output_half_width_mm = 4.5

X_input = np.linspace(
    -X_input_half_width_mm,
    X_input_half_width_mm,
    N_input,
)

U_spectral = np.linspace(
    -U_half_width_mm,
    U_half_width_mm,
    N_spectral,
)

X_output = np.linspace(
    -X_output_half_width_mm,
    X_output_half_width_mm,
    N_output,
)

# Расширенная расчётная сетка для обратного преобразования Фурье
N_output_calc = 65537
X_output_calc_half_width_mm = 32.0

X_output_calc = np.linspace(
    -X_output_calc_half_width_mm,
    X_output_calc_half_width_mm,
    N_output_calc,
)

dx_output_calc = X_output_calc[1] - X_output_calc[0]

crop_start = int(round(
    (X_output[0] - X_output_calc[0]) / dx_output_calc
))
crop_stop = crop_start + N_output

print("Входных точек:", N_input)
print("Спектральных точек:", N_spectral)
print("Сохраняемых выходных точек:", N_output)
print("Точек внутреннего расчёта:", N_output_calc)

print("Спектральная область:", U_spectral[[0, -1]], "мм")
print("Сохраняемая область:", X_output[[0, -1]], "мм")
print("Внутренняя область расчёта:", X_output_calc[[0, -1]], "мм")

Входных точек: 2049
Спектральных точек: 16385
Сохраняемых выходных точек: 9217
Точек внутреннего расчёта: 65537
Спектральная область: [-5.5  5.5] мм
Сохраняемая область: [-4.5  4.5] мм
Внутренняя область расчёта: [-32.  32.] мм


In [3]:
LABEL_COLS = [
    "label_amplitude_slit",
    "label_amplitude_binary",
    "label_phase_linear",
    "label_phase_sinusoidal",
]

FILTER_NAMES = [
    "amplitude_slit",
    "amplitude_binary",
    "phase_linear",
    "phase_sinusoidal",
]

FEATURE_COLS = [
    "output_energy",
    "output_centroid_mm",
    "output_rms_width_mm",
    "output_peak_intensity",
    "output_intensity_symmetry_error",
]

PARAMETER_COLS = [
    "alpha_rad_per_mm",
    "sigma_mm",
    "slit_width_mm",
    "slit_target_transmission",
    "slit_actual_transmission",
    "period_binary_mm",
    "duty_cycle",
    "binary_high",
    "binary_low",
    "phase_slope_rad_per_mm",
    "period_phase_sinusoidal_mm",
    "phase_depth_rad",
]


# Диапазоны исходных параметров и остальных фильтров.

ALPHA_RANGE = (5.0, 300.0)
SIGMA_RANGE = (0.02, 0.35)

BINARY_PERIOD_RANGE = (0.02, 0.04)
BINARY_DUTY_RANGE = (0.20, 0.80)
BINARY_HIGH_RANGE = (0.60, 1.00)
BINARY_LOW_RANGE = (0.00, 0.40)

PHASE_LINEAR_RANGE = (-300.0, 300.0)

PHASE_SIN_PERIOD_RANGE = (0.02, 0.04)
PHASE_DEPTH_RANGE = (0.10, 2.00)
SLIT_TARGET_TRANSMISSION_RANGE = (0.05, 0.95)


In [4]:
# Всё из вкр бака

def gaussian_sinus(x, alpha, sigma):
    """Гауссов пучок с синусной модуляцией: exp(-x²/σ²)·sin(αx)"""
    beam = np.exp(-x**2 / sigma**2)
    mod = np.sin(x * alpha)
    return beam * mod

def phase_sinusoidal(x, m, period):
    """Синусоидальная фазовая решётка: exp(i·m·sin(2πx/d))"""
    return np.exp(1j * m * np.sin(2 * np.pi * x / period))

def amplitude_binary(x, period, duty_cycle=0.5, high=1.0, low=0.0):
    """Бинарная амплитудная решётка."""
    # Положение внутри периода
    pos = np.mod(x, period)
    # Маска для "высокого" уровня
    mask = pos < (period * duty_cycle)
    res = np.where(mask, high, low)
    return res

# Простейшая функция – постоянный фазовый сдвиг (для фильтра)
def phase_modulator(phi):
    """Возвращает функцию, умножающую спектр на exp(i·phi)"""
    return lambda u: np.exp(1j * phi) * np.ones_like(u)

# Собственное быстрое преобразование Фурье (radix-2)
def next_pow2(n):
    """Ближайшая степень двойки, не меньшая n"""
    return 1 << (n - 1).bit_length()

def bit_reverse_copy(a):
    """Перестановка массива в двоично-инверсном порядке (in‑place)"""
    n = len(a)
    j = 0
    for i in range(1, n):
        bit = n >> 1
        while j & bit:
            j ^= bit
            bit >>= 1
        j ^= bit
        if i < j:
            a[i], a[j] = a[j], a[i]

def fft_radix2(a):
    """Прямое БПФ (in‑place) для массива длины N = 2^k"""
    bit_reverse_copy(a)
    n = len(a)
    step = 1
    while step < n:
        jump = step << 1
        # поворачивающие множители для текущего уровня
        wlen = np.exp(-1j * np.pi / step)
        for m in range(0, n, jump):
            w = 1.0 + 0.0j
            for k in range(step):
                u = a[m + k]
                t = w * a[m + step + k]
                a[m + k] = u + t
                a[m + step + k] = u - t
                w *= wlen
        step = jump

def ifft_radix2(a):
    """Обратное БПФ (in‑place) с делением на N"""
    a[:] = np.conj(a)
    fft_radix2(a)
    a[:] = np.conj(a) / len(a)

# Функции прямого и обратного преобразования Фурье с интерполяцией
def fourier_transform(x, u, func):
    """
    Прямое преобразование Фурье (4f-система) с использованием БПФ.
    x  – равномерная сетка входных координат,
    u  – равномерная сетка частот, на которой требуется спектр,
    func – комплексное поле на сетке x.
    Возвращает спектр G(u) на сетке u.
    """
    dx = x[1] - x[0]
    u_period = Wavelength * Focal_length / dx   # полный период по частоте

    # Определяем размер БПФ (степень двойки)
    Nfft = max(next_pow2(len(x)), 512)
    if len(u) > 1:
        du_req = u[1] - u[0]
        req_du = du_req / 4
        req_N = int(u_period / req_du)
        Nfft = max(Nfft, next_pow2(req_N))

    # Дополнение нулями
    padded = np.zeros(Nfft, dtype=complex)
    padded[:len(x)] = func

    # БПФ
    fft_radix2(padded)
    padded *= dx / (Wavelength * Focal_length)   # масштаб

    # Родная сетка частот (от 0 до u_period)
    du = u_period / Nfft
    k = np.arange(Nfft)
    u_native = k * du
    # Центрируем: отрицательные частоты переносим влево
    high_half = k > Nfft // 2
    u_native[high_half] -= u_period

    # Приводим запрошенные u в интервал [-u_period/2, u_period/2)
    u_mod = (u + u_period/2) % u_period - u_period/2

    # Дополнительный фазовый множитель (учитывает смещение апертуры)
    # В большинстве случаев не требуется, но оставлен для совместимости
    padded *= np.exp(1j * Wave_number * u_native * X_input_half_width_mm / Focal_length)

    # Интерполяция на запрошенную сетку
    sorted_idx = np.argsort(u_native)
    u_sorted = u_native[sorted_idx]
    spec_sorted = padded[sorted_idx]

    real_part = np.interp(u_mod, u_sorted, spec_sorted.real, left=0.0, right=0.0)
    imag_part = np.interp(u_mod, u_sorted, spec_sorted.imag, left=0.0, right=0.0)
    return real_part + 1j * imag_part

def inv_fourier_transform(spectrum, x_out, u_in):
    """
    Обратное преобразование Фурье (восстановление поля из спектра).
    spectrum – комплексный спектр на сетке u_in,
    x_out    – равномерная сетка выходных координат,
    u_in     – сетка частот, на которой задан спектр.
    Возвращает поле g(x) на сетке x_out.
    """
    dx_out = x_out[1] - x_out[0]
    Nx = len(x_out)
    M = max(next_pow2(Nx), 512)
    du_out = (Wavelength * Focal_length) / (M * dx_out)
    u_period_out = du_out * M

    half = M // 2
    u_native = (np.arange(M) - half) * du_out   # центрированная родная сетка

    # Интерполяция входного спектра на родную сетку БПФ
    real_spec = np.interp(u_native, u_in, np.real(spectrum), left=0.0, right=0.0)
    imag_spec = np.interp(u_native, u_in, np.imag(spectrum), left=0.0, right=0.0)
    U_centered = real_spec + 1j * imag_spec

    # Перестановка: нулевая частота в начало (ifftshift)
    U_fft_order = np.empty(M, dtype=complex)
    U_fft_order[:half] = U_centered[half:]
    U_fft_order[half:] = U_centered[:half]

    # Обратное БПФ
    ifft_radix2(U_fft_order)

    # Обратная перестановка (fftshift)
    field_centered = np.empty(M, dtype=complex)
    field_centered[:half] = U_fft_order[half:]
    field_centered[half:] = U_fft_order[:half]
    field_centered *= u_period_out

    # Интерполяция на выходную сетку x_out
    x_native = (np.arange(M) - half) * dx_out
    real_out = np.interp(x_out, x_native, field_centered.real, left=0.0, right=0.0)
    imag_out = np.interp(x_out, x_native, field_centered.imag, left=0.0, right=0.0)
    return real_out + 1j * imag_out

#Вспомогательная функция
def Fourier_func(u,x):
    Exp = np.exp(-1j * Wave_number * u * x / Focal_length)
    return Exp

#Прямое преобразование Фурье методом численного интергрирования
def Fourier(X, U, func):
    h = 2 * X.max() / X.size
    Result = np.zeros(U.size, dtype = complex)
    for i in range(X.size):
        for k in range(U.size):
            Result[k] += Fourier_func(U[k],X[i]) * func[i]
    Result = Result * h / (1 * Wavelength * Focal_length)

    return(Result)

#Вспомогательная обратная функция
def Inv_Fourier_func(u,x):
    iExp = np.exp(1j * Wave_number * u * x / Focal_length)
    return iExp

#Обратное преобразование Фурье методом численного интергрирования
def Inv_Fourier(func, X, U):
    h = 2 * U.max() / U.size
    Result = np.zeros(X.size, dtype = complex)
    for i in range(U.size):
        for k in range(X.size):
            Result[k] += Inv_Fourier_func(U[i],X[k]) * func[i]
    Result = Result * h

    return(Result)

In [5]:
def amplitude_slit(u, slit_width_mm):
    if not np.isfinite(slit_width_mm) or slit_width_mm <= 0:
        raise ValueError("Ширина щели должна быть положительной.")

    return (np.abs(np.asarray(u, dtype=float)) <= slit_width_mm / 2).astype(float)


def phase_linear(u, slope_rad_per_mm):
    return np.exp(1j * float(slope_rad_per_mm) * np.asarray(u, dtype=float))


def trapezoid_weights(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 1 or len(x) < 2:
        raise ValueError("Сетка должна содержать минимум две точки.")

    dx = np.diff(x)

    if not np.isfinite(x).all() or not np.all(dx > 0):
        raise ValueError("Сетка должна быть конечной и возрастающей.")

    weights = np.empty_like(x)
    weights[0] = dx[0] / 2
    weights[-1] = dx[-1] / 2
    weights[1:-1] = (x[2:] - x[:-2]) / 2

    return weights

def choose_slit_width(u, spectrum_before_slit, target_transmission):
    u = np.asarray(u, dtype=float)
    spectrum = np.asarray(spectrum_before_slit, dtype=complex)

    if u.ndim != 1 or spectrum.ndim != 1 or u.shape != spectrum.shape:
        raise ValueError("Сетка и спектр должны иметь одинаковую длину.")

    if not np.isfinite(u).all() or not np.isfinite(spectrum).all():
        raise ValueError("Сетка или спектр содержат NaN или Inf.")

    if not 0 < target_transmission < 1:
        raise ValueError("Целевое пропускание должно быть между 0 и 1.")

    weights = trapezoid_weights(u)
    weighted_power = weights * np.abs(spectrum) ** 2
    total_power = float(np.sum(weighted_power))

    if not np.isfinite(total_power) or total_power <= 0:
        raise ValueError("Спектральная мощность должна быть положительной.")

    # Группируем точки по расстоянию от центра спектра.
    abs_u = np.round(np.abs(u), decimals=12)
    order = np.argsort(abs_u)

    abs_sorted = abs_u[order]
    power_sorted = weighted_power[order]

    abs_levels, first_positions = np.unique(
        abs_sorted, return_index=True
    )

    grouped_power = np.add.reduceat(power_sorted, first_positions)
    cumulative_power = np.cumsum(grouped_power)
    target_power = target_transmission * total_power

    selected = np.searchsorted(cumulative_power, target_power)
    selected = min(int(selected), len(abs_levels) - 1)

    half_width = float(abs_levels[selected])
    slit_width_mm = 2 * half_width

    mask = abs_u <= half_width
    actual_transmission = float(
        np.sum(weighted_power[mask]) / total_power
    )

    return slit_width_mm, actual_transmission


def calculate_output_features(E_out, x_out=X_output):
    E_out = np.asarray(E_out, dtype=complex)
    x_out = np.asarray(x_out, dtype=float)

    if E_out.ndim != 1 or E_out.shape != x_out.shape:
        raise ValueError("Поле и координатная сетка должны совпадать по форме.")

    if not np.isfinite(E_out).all() or not np.isfinite(x_out).all():
        raise ValueError("Поле или координаты содержат NaN или Inf.")

    intensity = np.abs(E_out) ** 2
    energy = float(np.trapezoid(intensity, x_out))

    if not np.isfinite(energy) or energy <= 0:
        raise ValueError("Энергия выходного поля должна быть положительной.")

    centroid = float(np.trapezoid(intensity * x_out, x_out) / energy)

    variance = float(
        np.trapezoid(intensity * (x_out - centroid) ** 2, x_out) / energy
    )

    return {
        "output_energy": energy,
        "output_centroid_mm": centroid,
        "output_rms_width_mm": float(np.sqrt(max(variance, 0))),
        "output_peak_intensity": float(np.max(intensity)),
        "output_intensity_symmetry_error": float(
            np.sum(np.abs(intensity - intensity[::-1]))
            / (np.sum(intensity) + 1e-300)
        ),
    }


# Ключевая функция
def simulate_one(params, n_spectral=N_spectral, fixed_slit_width_mm=None):
    labels = np.asarray(params["labels"], dtype=int)

    if labels.shape != (4,) or not np.isin(labels, [0, 1]).all():
        raise ValueError("Нужны четыре бинарные метки фильтров.")

    u = np.linspace(-U_half_width_mm, U_half_width_mm, int(n_spectral))

    # Входное поле и исходное прямое преобразование Фурье.
    input_field = gaussian_sinus(
        X_input,
        float(params["alpha_rad_per_mm"]),
        float(params["sigma_mm"]),
    )
    spectrum = fourier_transform(X_input, u, input_field)

    # Фильтры, кроме амплитудной щели.
    H_other = np.ones_like(spectrum, dtype=complex)

    if labels[1]:
        H_other *= amplitude_binary(
            u,
            float(params["period_binary_mm"]),
            float(params["duty_cycle"]),
            float(params["binary_high"]),
            float(params["binary_low"]),
        )

    if labels[2]:
        H_other *= phase_linear(
            u, float(params["phase_slope_rad_per_mm"])
        )

    if labels[3]:
        H_other *= phase_sinusoidal(
            u,
            float(params["phase_depth_rad"]),
            float(params["period_phase_sinusoidal_mm"]),
        )

    spectrum_before_slit = spectrum * H_other
    power_before_slit = float(
        np.trapezoid(np.abs(spectrum_before_slit) ** 2, u)
    )

    # Амплитудная щель.
    if labels[0]:
        if fixed_slit_width_mm is None:
            slit_width_mm, _ = choose_slit_width(
                u,
                spectrum_before_slit,
                float(params["slit_target_transmission"]),
            )
        else:
            slit_width_mm = float(fixed_slit_width_mm)

        slit_mask = amplitude_slit(u, slit_width_mm)
        filtered_spectrum = spectrum_before_slit * slit_mask

        power_after_slit = float(
            np.trapezoid(np.abs(filtered_spectrum) ** 2, u)
        )
        actual_transmission = (
            power_after_slit / power_before_slit
            if power_before_slit > 0 else np.nan
        )
        slit_mask_symmetric = np.array_equal(slit_mask, slit_mask[::-1])

    else:
        slit_width_mm = np.nan
        filtered_spectrum = spectrum_before_slit.copy()
        power_after_slit = power_before_slit
        actual_transmission = np.nan
        slit_mask_symmetric = np.nan

    # Исходное обратное преобразование на расширенной сетке.
    output_field_full = inv_fourier_transform(
        filtered_spectrum, X_output_calc, u
    )

    # Оставляем исходную область выходных координат.
    output_field = output_field_full[crop_start:crop_stop]

    if len(output_field) != N_output:
        raise ValueError("Неверная длина выходного поля после обрезки.")

    features = calculate_output_features(output_field, X_output)

    return {
        "u_spectral_mm": u,
        "input_field": input_field,
        "spectrum_unfiltered": spectrum,
        "spectrum_before_slit": spectrum_before_slit,
        "filtered_spectrum": filtered_spectrum,
        "output_field": output_field,
        "output_field_full": output_field_full,
        "slit_width_mm": float(slit_width_mm),
        "slit_actual_transmission": float(actual_transmission),
        "slit_mask_symmetric": slit_mask_symmetric,
        "spectral_power_before_slit": power_before_slit,
        "spectral_power_after_slit": power_after_slit,
        "features": features,
    }


## Генератор во всей красе

In [6]:

def draw_parameters(labels, rng):
    labels = np.asarray(labels, dtype=int)

    params = {
        "labels": labels.copy(),
        "alpha_rad_per_mm": float(rng.uniform(*ALPHA_RANGE)),
        "sigma_mm": float(rng.uniform(*SIGMA_RANGE)),
        "slit_width_mm": np.nan,
        "slit_target_transmission": np.nan,
        "period_binary_mm": np.nan,
        "duty_cycle": np.nan,
        "binary_high": np.nan,
        "binary_low": np.nan,
        "phase_slope_rad_per_mm": np.nan,
        "period_phase_sinusoidal_mm": np.nan,
        "phase_depth_rad": np.nan,
    }

    if labels[0]:
        params["slit_target_transmission"] = float(
            rng.uniform(*SLIT_TARGET_TRANSMISSION_RANGE)
        )

    if labels[1]:
        params["period_binary_mm"] = float(rng.uniform(*BINARY_PERIOD_RANGE))
        params["duty_cycle"] = float(rng.uniform(*BINARY_DUTY_RANGE))
        params["binary_high"] = float(rng.uniform(*BINARY_HIGH_RANGE))
        params["binary_low"] = float(rng.uniform(*BINARY_LOW_RANGE))

    if labels[2]:
        params["phase_slope_rad_per_mm"] = float(
            rng.uniform(*PHASE_LINEAR_RANGE)
        )

    if labels[3]:
        params["period_phase_sinusoidal_mm"] = float(
            rng.uniform(*PHASE_SIN_PERIOD_RANGE)
        )
        params["phase_depth_rad"] = float(rng.uniform(*PHASE_DEPTH_RANGE))

    return params


In [7]:

def generate_multilabel_dataset(
    n_samples_total=200,
    output_dir="dataset_multilabel_adaptive_slit_200",
    n_spectral=N_spectral,
    random_seed=20261025,
):
    n_samples_total = int(n_samples_total)
    n_spectral = int(n_spectral)

    if n_samples_total < 15:
        raise ValueError("Нужно минимум 15 примеров для всех комбинаций фильтров.")
    if n_spectral < 3:
        raise ValueError("n_spectral должен быть не меньше 3.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(random_seed)

    # Равномерно распределяем примеры между 15 комбинациями.
    masks = np.arange(1, 16)
    counts = np.full(15, n_samples_total // 15, dtype=int)
    counts[:n_samples_total % 15] += 1

    sample_masks = np.concatenate([
        np.repeat(mask, count)
        for mask, count in zip(masks, counts)
    ])
    rng.shuffle(sample_masks)

    rows = []
    sample_ids = []
    labels_all = []
    Eout_real_all = []
    Eout_imag_all = []
    spectrum_amplitude_all = []
    spectrum_phase_all = []
    slit_target_all = []
    slit_actual_all = []
    power_before_all = []
    power_after_all = []

    start_time = time.perf_counter()

    for i, mask in enumerate(sample_masks):
        labels = np.array([(int(mask) >> j) & 1 for j in range(4)])
        params = draw_parameters(labels, rng)
        result = simulate_one(params, n_spectral=n_spectral)

        sample_id = f"ml_sample_{i:06d}"

        # Сохраняем выходное поле в float32.
        E_real = result["output_field"].real.astype(np.float32)
        E_imag = result["output_field"].imag.astype(np.float32)
        E_saved = E_real.astype(np.float64) + 1j * E_imag.astype(np.float64)

        features = calculate_output_features(E_saved, X_output)
        spectrum = result["filtered_spectrum"]

        active_names = [
            FILTER_NAMES[j] for j in range(4) if labels[j] == 1
        ]

        row = {
            "sample_id": sample_id,
            "filter_mask": int(mask),
            "filter_combination": "+".join(active_names),
            "alpha_rad_per_mm": params["alpha_rad_per_mm"],
            "sigma_mm": params["sigma_mm"],
            "slit_width_mm": result["slit_width_mm"],
            "slit_target_transmission": params["slit_target_transmission"],
            "slit_actual_transmission": result["slit_actual_transmission"],
            "period_binary_mm": params["period_binary_mm"],
            "duty_cycle": params["duty_cycle"],
            "binary_high": params["binary_high"],
            "binary_low": params["binary_low"],
            "phase_slope_rad_per_mm": params["phase_slope_rad_per_mm"],
            "period_phase_sinusoidal_mm": params["period_phase_sinusoidal_mm"],
            "phase_depth_rad": params["phase_depth_rad"],
            "spectral_power_before_slit": result["spectral_power_before_slit"],
            "spectral_power_after_slit": result["spectral_power_after_slit"],
            "n_spectral": n_spectral,
            **features,
        }

        for j, name in enumerate(LABEL_COLS):
            row[name] = int(labels[j])

        rows.append(row)
        sample_ids.append(sample_id)
        labels_all.append(labels)
        Eout_real_all.append(E_real)
        Eout_imag_all.append(E_imag)
        spectrum_amplitude_all.append(np.abs(spectrum).astype(np.float32))
        spectrum_phase_all.append(np.angle(spectrum).astype(np.float32))
        slit_target_all.append(params["slit_target_transmission"])
        slit_actual_all.append(result["slit_actual_transmission"])
        power_before_all.append(result["spectral_power_before_slit"])
        power_after_all.append(result["spectral_power_after_slit"])

        if (i + 1) % 10 == 0 or i + 1 == n_samples_total:
            elapsed = time.perf_counter() - start_time
            print(f"Готово: {i + 1}/{n_samples_total}, время: {elapsed:.1f} с")

    df = pd.DataFrame(rows)

    arrays = {
        "sample_id": np.asarray(sample_ids, dtype="U32"),
        "filter_labels": np.asarray(labels_all, dtype=np.uint8),
        "Eout_real": np.stack(Eout_real_all),
        "Eout_imag": np.stack(Eout_imag_all),
        "filtered_spectrum_amplitude": np.stack(spectrum_amplitude_all),
        "filtered_spectrum_phase": np.stack(spectrum_phase_all),
        "x_output_mm": X_output.astype(np.float64),
        "u_spectral_mm": np.linspace(
            -U_half_width_mm, U_half_width_mm, n_spectral
        ),
        "slit_target_transmission": np.asarray(slit_target_all, dtype=np.float32),
        "slit_actual_transmission": np.asarray(slit_actual_all, dtype=np.float32),
        "spectral_power_before_slit": np.asarray(power_before_all, dtype=np.float64),
        "spectral_power_after_slit": np.asarray(power_after_all, dtype=np.float64),
    }

    # Сохраняем таблицу и численные массивы.
    df.to_csv(output_dir / "dataset_params.csv", index=False, encoding="utf-8-sig")
    np.savez_compressed(output_dir / "dataset_vectors.npz", **arrays)

    elapsed = time.perf_counter() - start_time

    config = {
        "generator_version": "adaptive_slit",
        "random_seed": random_seed,
        "n_samples_total": n_samples_total,
        "n_spectral": n_spectral,
        "wavelength_mm": float(Wavelength),
        "focal_length_mm": float(Focal_length),
        "input_half_width_mm": float(X_input_half_width_mm),
        "output_half_width_mm": float(X_output_half_width_mm),
        "n_input": int(N_input),
        "n_output": int(N_output),
        "alpha_range_rad_per_mm": list(ALPHA_RANGE),
        "sigma_range_mm": list(SIGMA_RANGE),
        "slit_target_transmission_range": list(SLIT_TARGET_TRANSMISSION_RANGE),
        "filter_names_in_label_order": FILTER_NAMES,
        "filter_mask_bit_weights": {
            name: 1 << j for j, name in enumerate(FILTER_NAMES)
        },
        "fourier_functions": [
            "fourier_transform", "inv_fourier_transform",
            "fft_radix2", "ifft_radix2",
        ],
        "generation_seconds": float(elapsed),
    }

    with open(output_dir / "dataset_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    print("\nГенерация завершена.")
    print("Строк:", len(df))
    print("Комбинаций фильтров:", df["filter_combination"].nunique())
    print(f"Время: {elapsed:.1f} с")
    print("Папка:", output_dir.resolve())

    return df, arrays


print("Генератор определён.")


Генератор определён.


## Тесты что генератор вообще работает

In [8]:
DATASET_DIR = Path("Main_dataset_genv1_2000")

df_dataset, arrays_dataset = generate_multilabel_dataset(
    n_samples_total=2000,
    output_dir=str(DATASET_DIR),
    n_spectral=16385,
    random_seed=20261025,
)

display(
    df_dataset["filter_combination"]
    .value_counts()
    .sort_index()
    .rename("n_samples")
    .to_frame()
)


Готово: 10/2000, время: 26.6 с
Готово: 20/2000, время: 52.0 с
Готово: 30/2000, время: 77.0 с
Готово: 40/2000, время: 102.2 с
Готово: 50/2000, время: 127.5 с
Готово: 60/2000, время: 152.8 с
Готово: 70/2000, время: 178.3 с
Готово: 80/2000, время: 203.6 с
Готово: 90/2000, время: 228.7 с
Готово: 100/2000, время: 254.2 с
Готово: 110/2000, время: 279.2 с
Готово: 120/2000, время: 304.5 с
Готово: 130/2000, время: 329.6 с
Готово: 140/2000, время: 354.7 с
Готово: 150/2000, время: 380.0 с
Готово: 160/2000, время: 405.0 с
Готово: 170/2000, время: 430.5 с
Готово: 180/2000, время: 455.7 с
Готово: 190/2000, время: 481.0 с
Готово: 200/2000, время: 506.1 с
Готово: 210/2000, время: 531.3 с
Готово: 220/2000, время: 556.5 с
Готово: 230/2000, время: 581.6 с
Готово: 240/2000, время: 606.9 с
Готово: 250/2000, время: 632.1 с
Готово: 260/2000, время: 657.3 с
Готово: 270/2000, время: 682.5 с
Готово: 280/2000, время: 707.6 с
Готово: 290/2000, время: 732.8 с
Готово: 300/2000, время: 757.8 с
Готово: 310/2000, врем

,n_samples
filter_combination,
amplitude_binary,134
amplitude_binary+phase_linear,133
amplitude_binary+phase_linear+phase_sinusoidal,133
amplitude_binary+phase_sinusoidal,133
amplitude_slit,134
amplitude_slit+amplitude_binary,134
amplitude_slit+amplitude_binary+phase_linear,133
amplitude_slit+amplitude_binary+phase_linear+phase_sinusoidal,133
amplitude_slit+amplitude_binary+phase_sinusoidal,133


In [9]:
DATASET_DIR = Path("Main_dataset_genv1_2000")
CSV_PATH = DATASET_DIR / "dataset_params.csv"
NPZ_PATH = DATASET_DIR / "dataset_vectors.npz"
CONFIG_PATH = DATASET_DIR / "dataset_config.json"

N_EXPECTED = 2000

# 1. Проверяем, что файлы существуют и читаются
for path in [CSV_PATH, NPZ_PATH, CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

df = pd.read_csv(CSV_PATH)

with np.load(NPZ_PATH, allow_pickle=False) as data:
    arrays = {name: data[name] for name in data.files}

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = json.load(file)

print("Все три файла читаются.")

# 2. Проверяем количество примеров
if len(df) != N_EXPECTED:
    raise ValueError(f"Ожидалось {N_EXPECTED} строк, получено {len(df)}.")

if len(arrays["sample_id"]) != N_EXPECTED:
    raise ValueError("Количество примеров в NPZ не совпадает с CSV.")

if len(arrays["filter_labels"]) != N_EXPECTED:
    raise ValueError("Неверное количество меток фильтров.")

print(f"Количество примеров: {len(df)}.")

# 3. Проверяем комбинации фильтров
combination_counts = df["filter_combination"].value_counts()

if len(combination_counts) != 15:
    raise ValueError(f"Ожидалось 15 комбинаций, найдено {len(combination_counts)}.")

if combination_counts.max() - combination_counts.min() > 1:
    raise ValueError("Примеры распределены по комбинациям неравномерно.")

print("Все 15 комбинаций присутствуют.")
print("Количество примеров по комбинациям:")
display(combination_counts.sort_index().rename("Количество").to_frame())

# 4. Проверяем диапазоны случайных параметров
range_checks = {
    "alpha_rad_per_mm": (ALPHA_RANGE, np.ones(len(df), dtype=bool)),
    "sigma_mm": (SIGMA_RANGE, np.ones(len(df), dtype=bool)),
    "slit_target_transmission": (
        SLIT_TARGET_TRANSMISSION_RANGE,
        df[LABEL_COLS[0]] == 1,
    ),
    "period_binary_mm": (
        BINARY_PERIOD_RANGE,
        df[LABEL_COLS[1]] == 1,
    ),
    "duty_cycle": (
        BINARY_DUTY_RANGE,
        df[LABEL_COLS[1]] == 1,
    ),
    "binary_high": (
        BINARY_HIGH_RANGE,
        df[LABEL_COLS[1]] == 1,
    ),
    "binary_low": (
        BINARY_LOW_RANGE,
        df[LABEL_COLS[1]] == 1,
    ),
    "phase_slope_rad_per_mm": (
        PHASE_LINEAR_RANGE,
        df[LABEL_COLS[2]] == 1,
    ),
    "period_phase_sinusoidal_mm": (
        PHASE_SIN_PERIOD_RANGE,
        df[LABEL_COLS[3]] == 1,
    ),
    "phase_depth_rad": (
        PHASE_DEPTH_RANGE,
        df[LABEL_COLS[3]] == 1,
    ),
}

for column, (limits, active_rows) in range_checks.items():
    values = df.loc[active_rows, column].dropna()
    low, high = limits

    if values.empty:
        raise ValueError(f"Нет значений для проверки параметра {column}.")

    if not values.between(low, high).all():
        raise ValueError(
            f"Параметр {column} вышел за диапазон [{low}, {high}]."
        )

print("Все случайные параметры находятся в заданных диапазонах.")

# 5. Проверяем размерности сохранённых массивов
expected_shapes = {
    "Eout_real": (N_EXPECTED, N_output),
    "Eout_imag": (N_EXPECTED, N_output),
    "filtered_spectrum_amplitude": (N_EXPECTED, N_spectral),
    "filtered_spectrum_phase": (N_EXPECTED, N_spectral),
}

for name, expected_shape in expected_shapes.items():
    if name not in arrays:
        raise ValueError(f"В NPZ отсутствует массив {name}.")

    if arrays[name].shape != expected_shape:
        raise ValueError(
            f"{name}: ожидалась форма {expected_shape}, "
            f"получена {arrays[name].shape}."
        )

# 6. Проверяем основные массивы на NaN и бесконечности
for name in [
    "Eout_real",
    "Eout_imag",
    "filtered_spectrum_amplitude",
    "filtered_spectrum_phase",
]:
    if not np.isfinite(arrays[name]).all():
        raise ValueError(f"В массиве {name} есть NaN или бесконечные значения.")

# 7. Проверяем, что признаки рассчитаны и конечны
feature_columns = [
    "output_energy",
    "output_centroid_mm",
    "output_rms_width_mm",
    "output_peak_intensity",
    "output_intensity_symmetry_error",
]

for column in feature_columns:
    if column not in df.columns:
        raise ValueError(f"В CSV отсутствует признак {column}.")

    if not np.isfinite(df[column].to_numpy(dtype=float)).all():
        raise ValueError(f"В признаке {column} есть NaN или бесконечные значения.")

print("Размеры массивов корректны, признаки рассчитаны.")
print("Все простые проверки датасета пройдены.")


Все три файла читаются.
Количество примеров: 2000.
Все 15 комбинаций присутствуют.
Количество примеров по комбинациям:


,Количество
filter_combination,
amplitude_binary,134
amplitude_binary+phase_linear,133
amplitude_binary+phase_linear+phase_sinusoidal,133
amplitude_binary+phase_sinusoidal,133
amplitude_slit,134
amplitude_slit+amplitude_binary,134
amplitude_slit+amplitude_binary+phase_linear,133
amplitude_slit+amplitude_binary+phase_linear+phase_sinusoidal,133
amplitude_slit+amplitude_binary+phase_sinusoidal,133


Все случайные параметры находятся в заданных диапазонах.
Размеры массивов корректны, признаки рассчитаны.
Все простые проверки датасета пройдены.
